In [ ]:
!pip install pandas numpy matplotlib tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_adm = pd.read_csv('/content/drive/MyDrive/NarrativeGuard/mimic-discharge-summaries/admissions.csv')
df_adm.columns = df_adm.columns.str.upper()

df_adm['ADMITTIME'] = pd.to_datetime(df_adm.ADMITTIME, format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')
df_adm['DISCHTIME'] = pd.to_datetime(df_adm.DISCHTIME, format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')
df_adm['DEATHTIME'] = pd.to_datetime(df_adm.DEATHTIME, format = '%Y-%m-%d %H:%M:%S', errors = 'coerce')

df_adm = df_adm.sort_values(['SUBJECT_ID','ADMITTIME'])
df_adm = df_adm.reset_index(drop = True)
df_adm['NEXT_ADMITTIME'] = df_adm.groupby('SUBJECT_ID').ADMITTIME.shift(-1)
df_adm['NEXT_ADMISSION_TYPE'] = df_adm.groupby('SUBJECT_ID').ADMISSION_TYPE.shift(-1)

rows = df_adm.NEXT_ADMISSION_TYPE == 'ELECTIVE'
df_adm.loc[rows,'NEXT_ADMITTIME'] = pd.NaT
df_adm.loc[rows,'NEXT_ADMISSION_TYPE'] = np.nan

df_adm = df_adm.sort_values(['SUBJECT_ID','ADMITTIME'])

#When we filter out the "ELECTIVE", we need to correct the next admit time for these admissions since there might be 'emergency' next admit after "ELECTIVE"
df_adm[['NEXT_ADMITTIME','NEXT_ADMISSION_TYPE']] = df_adm.groupby(['SUBJECT_ID'])[['NEXT_ADMITTIME','NEXT_ADMISSION_TYPE']].fillna(method = 'bfill')
df_adm['DAYS_NEXT_ADMIT']=  (df_adm.NEXT_ADMITTIME - df_adm.DISCHTIME).dt.total_seconds()/(24*60*60)
df_adm['OUTPUT_LABEL'] = (df_adm.DAYS_NEXT_ADMIT < 30).astype('int')
### filter out newborn and death
df_adm = df_adm[df_adm['ADMISSION_TYPE']!='NEWBORN']
df_adm = df_adm[df_adm.DEATHTIME.isnull()]
df_adm['DURATION'] = (df_adm['DISCHTIME']-df_adm['ADMITTIME']).dt.total_seconds()/(24*60*60)

print(df_adm.head(10))

/tmp/ipykernel_18827/3466273870.py:20: FutureWarning: DataFrameGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use DataFrame.fillna instead
  df_adm[['NEXT_ADMITTIME','NEXT_ADMISSION_TYPE']] = df_adm.groupby(['SUBJECT_ID'])[['NEXT_ADMITTIME','NEXT_ADMISSION_TYPE']].fillna(method = 'bfill')
/tmp/ipykernel_18827/3466273870.py:20: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_adm[['NEXT_ADMITTIME','NEXT_ADMISSION_TYPE']] = df_adm.groupby(['SUBJECT_ID'])[['NEXT_ADMITTIME','NEXT_ADMISSION_TYPE']].fillna(method = 'bfill')
/tmp/ipykernel_18827/3466273870.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set

   SUBJECT_ID   HADM_ID           ADMITTIME           DISCHTIME DEATHTIME  \
0    10000032  22595853 2180-05-06 22:23:00 2180-05-07 17:15:00       NaT   
1    10000032  22841357 2180-06-26 18:27:00 2180-06-27 18:49:00       NaT   
2    10000032  29079034 2180-07-23 12:35:00 2180-07-25 17:55:00       NaT   
3    10000032  25742920 2180-08-05 23:44:00 2180-08-07 17:50:00       NaT   
4    10000068  25022803 2160-03-03 23:16:00 2160-03-04 06:26:00       NaT   
5    10000084  23052089 2160-11-21 01:56:00 2160-11-25 14:52:00       NaT   
6    10000084  29888819 2160-12-28 05:11:00 2160-12-28 16:07:00       NaT   
7    10000108  27250926 2163-09-27 23:17:00 2163-09-28 09:04:00       NaT   
8    10000117  22927623 2181-11-15 02:05:00 2181-11-15 14:52:00       NaT   
9    10000117  27988844 2183-09-18 18:10:00 2183-09-21 16:30:00       NaT   

      ADMISSION_TYPE ADMIT_PROVIDER_ID      ADMISSION_LOCATION  \
0             URGENT            P49AFC  TRANSFER FROM HOSPITAL   
1           EW EMER.

In [ ]:
df_discharge = pd.read_csv('/content/drive/MyDrive/NarrativeGuard/mimic-discharge-summaries/discharge.csv')
df_discharge.columns = df_discharge.columns.str.upper()

df_discharge = df_discharge.sort_values(by=['SUBJECT_ID','HADM_ID','CHARTTIME'])
df_adm_disch = pd.merge(df_adm[['SUBJECT_ID','HADM_ID','ADMITTIME','ADMISSION_LOCATION', 'DISCHARGE_LOCATION', 'INSURANCE','LANGUAGE', 'MARITAL_STATUS', 'RACE', 'DISCHTIME','DAYS_NEXT_ADMIT','NEXT_ADMITTIME','ADMISSION_TYPE','DEATHTIME','OUTPUT_LABEL','DURATION']],
                        df_discharge[['SUBJECT_ID','HADM_ID','CHARTTIME','NOTE_SEQ','TEXT']],
                        on = ['SUBJECT_ID','HADM_ID'],
                        how = 'left')

df_adm_disch['ADMITTIME_C'] = df_adm_disch['ADMITTIME'].apply(lambda x: str(x).split(' ')[0])
df_adm_disch['ADMITTIME_C'] = pd.to_datetime(df_adm_disch['ADMITTIME_C'], format = '%Y-%m-%d', errors = 'coerce')
df_adm_disch['CHARTTIME'] = df_adm_disch['CHARTTIME'].apply(lambda x: str(x).split(' ')[0])

df_adm_disch['CHARTDATE_C'] = pd.to_datetime(df_adm_disch['CHARTTIME'], format = '%Y-%m-%d', errors = 'coerce')

# multiple discharge summary for one admission -> after examination -> replicated summary -> replace with the last one
df_adm_disch = (df_adm_disch.groupby(['SUBJECT_ID','HADM_ID']).nth(-1)).reset_index()
df_adm_disch=df_adm_disch[df_adm_disch['TEXT'].notnull()]


KeyboardInterrupt: 

In [ ]:
print(df_adm_disch.head(5))

   index  SUBJECT_ID   HADM_ID           ADMITTIME      ADMISSION_LOCATION  \
0      0    10000032  22595853 2180-05-06 22:23:00  TRANSFER FROM HOSPITAL   
1      1    10000032  22841357 2180-06-26 18:27:00          EMERGENCY ROOM   
2      2    10000032  29079034 2180-07-23 12:35:00          EMERGENCY ROOM   
3      3    10000032  25742920 2180-08-05 23:44:00          EMERGENCY ROOM   
5      5    10000084  23052089 2160-11-21 01:56:00   WALK-IN/SELF REFERRAL   

  DISCHARGE_LOCATION INSURANCE LANGUAGE MARITAL_STATUS   RACE  \
0               HOME  Medicaid  English        WIDOWED  WHITE   
1               HOME  Medicaid  English        WIDOWED  WHITE   
2               HOME  Medicaid  English        WIDOWED  WHITE   
3            HOSPICE  Medicaid  English        WIDOWED  WHITE   
5   HOME HEALTH CARE  Medicare  English        MARRIED  WHITE   

            DISCHTIME  DAYS_NEXT_ADMIT      NEXT_ADMITTIME ADMISSION_TYPE  \
0 2180-05-07 17:15:00        50.050000 2180-06-26 18:27:00     

In [ ]:
df_adm_disch.to_csv('/content/drive/MyDrive/adm_disch_comb.csv')

In [26]:
import pandas as pd

# Load df_adm_disch from the saved CSV
file_path = '/content/drive/MyDrive/adm_disch_comb.csv'
df_adm_disch = pd.read_csv(file_path)

print(f'Loaded {len(df_adm_disch)} rows from {file_path}.')
display(df_adm_disch.head())

Loaded 323603 rows from /content/drive/MyDrive/adm_disch_comb.csv.


,Unnamed: 0,index,SUBJECT_ID,HADM_ID,ADMITTIME,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,MARITAL_STATUS,...,NEXT_ADMITTIME,ADMISSION_TYPE,DEATHTIME,OUTPUT_LABEL,DURATION,CHARTTIME,NOTE_SEQ,TEXT,ADMITTIME_C,CHARTDATE_C
0,0,0,10000032,22595853,2180-05-06 22:23:00,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,...,2180-06-26 18:27:00,URGENT,NaN,0,0.786111,2180-05-07,21.0,name: _ unit no: _ ...,2180-05-06,2180-05-07
1,1,1,10000032,22841357,2180-06-26 18:27:00,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,...,2180-07-23 12:35:00,EW EMER.,NaN,1,1.015278,2180-06-27,22.0,name: _ unit no: _ ...,2180-06-26,2180-06-27
2,2,2,10000032,29079034,2180-07-23 12:35:00,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,...,2180-08-05 23:44:00,EW EMER.,NaN,1,2.222222,2180-07-25,23.0,name: _ unit no: _ ...,2180-07-23,2180-07-25
3,3,3,10000032,25742920,2180-08-05 23:44:00,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,...,NaN,EW EMER.,NaN,0,1.754167,2180-08-07,24.0,name: _ unit no: _ ...,2180-08-05,2180-08-07
4,5,5,10000084,23052089,2160-11-21 01:56:00,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,English,MARRIED,...,2160-12-28 05:11:00,EW EMER.,NaN,0,4.538889,2160-11-25,17.0,name: _ unit no: _ _...,2160-11-21,2160-11-25


In [14]:
import pandas as pd

file_path = '/content/drive/MyDrive/adm_disch_comb.csv'

print('--- Method 1: Peek at the first 5 rows ---')
df_head = pd.read_csv(file_path, nrows=5)
print(f'Rows loaded: {len(df_head)}')
display(df_head)

print('\n--- Method 2: See a random row from the middle ---')
# Reading row 10,000
df_middle = pd.read_csv(file_path, skiprows=10000, nrows=1, names=['Unnamed: 0', 'index', 'SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'ADMISSION_LOCATION', 'DISCHARGE_LOCATION', 'INSURANCE', 'LANGUAGE', 'MARITAL_STATUS', 'RACE', 'DISCHTIME', 'DAYS_NEXT_ADMIT', 'NEXT_ADMITTIME', 'ADMISSION_TYPE', 'DEATHTIME', 'OUTPUT_LABEL', 'DURATION', 'CHARTTIME', 'NOTE_SEQ', 'TEXT', 'ADMITTIME_C', 'CHARTDATE_C'])
print(f'Middle row ID: {df_middle["HADM_ID"].iloc[0]}')
display(df_middle)

print('\n--- Method 3: Get Metadata ---')
df_structure = pd.read_csv(file_path, nrows=0)
print(f'Columns: {list(df_structure.columns)}')

--- Method 1: Peek at the first 5 rows ---
Rows loaded: 5


,Unnamed: 0,index,SUBJECT_ID,HADM_ID,ADMITTIME,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,MARITAL_STATUS,...,NEXT_ADMITTIME,ADMISSION_TYPE,DEATHTIME,OUTPUT_LABEL,DURATION,CHARTTIME,NOTE_SEQ,TEXT,ADMITTIME_C,CHARTDATE_C
0,0,0,10000032,22595853,2180-05-06 22:23:00,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,...,2180-06-26 18:27:00,URGENT,NaN,0,0.786111,2180-05-07,21.0,name: _ unit no: _ ...,2180-05-06,2180-05-07
1,1,1,10000032,22841357,2180-06-26 18:27:00,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,...,2180-07-23 12:35:00,EW EMER.,NaN,1,1.015278,2180-06-27,22.0,name: _ unit no: _ ...,2180-06-26,2180-06-27
2,2,2,10000032,29079034,2180-07-23 12:35:00,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,...,2180-08-05 23:44:00,EW EMER.,NaN,1,2.222222,2180-07-25,23.0,name: _ unit no: _ ...,2180-07-23,2180-07-25
3,3,3,10000032,25742920,2180-08-05 23:44:00,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,...,NaN,EW EMER.,NaN,0,1.754167,2180-08-07,24.0,name: _ unit no: _ ...,2180-08-05,2180-08-07
4,5,5,10000084,23052089,2160-11-21 01:56:00,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,English,MARRIED,...,2160-12-28 05:11:00,EW EMER.,NaN,0,4.538889,2160-11-25,17.0,name: _ unit no: _ _...,2160-11-21,2160-11-25



--- Method 2: See a random row from the middle ---
Middle row ID: 26774424


,Unnamed: 0,index,SUBJECT_ID,HADM_ID,ADMITTIME,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,MARITAL_STATUS,...,NEXT_ADMITTIME,ADMISSION_TYPE,DEATHTIME,OUTPUT_LABEL,DURATION,CHARTTIME,NOTE_SEQ,TEXT,ADMITTIME_C,CHARTDATE_C
0,16466,16466,10320090,26774424,2178-04-05 00:00:00,TRANSFER FROM HOSPITAL,HOME,Medicare,English,SINGLE,...,2178-04-20 04:34:00,URGENT,NaN,1,7.656944,2178-04-12,20.0,name: _ unit no: _ _ ...,2178-04-05,2178-04-12



--- Method 3: Get Metadata ---
Columns: ['Unnamed: 0', 'index', 'SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'ADMISSION_LOCATION', 'DISCHARGE_LOCATION', 'INSURANCE', 'LANGUAGE', 'MARITAL_STATUS', 'RACE', 'DISCHTIME', 'DAYS_NEXT_ADMIT', 'NEXT_ADMITTIME', 'ADMISSION_TYPE', 'DEATHTIME', 'OUTPUT_LABEL', 'DURATION', 'CHARTTIME', 'NOTE_SEQ', 'TEXT', 'ADMITTIME_C', 'CHARTDATE_C']


In [21]:
import re
import pandas as pd
from tqdm import tqdm

def preprocess1(x):
    y=re.sub(r'\[(.*?)\]','',x) #remove de-identified brackets
    y=re.sub(r'[0-9]+\.','',y) #remove 1.2. since the segmenter segments based on this
    y=re.sub(r'dr\.','doctor',y)
    y=re.sub(r'm\.d\.','md',y)
    y=re.sub('admission date:','',y)
    y=re.sub('discharge date:','',y)
    y=re.sub('--|__|==','',y)
    return y

def preprocessing(df_less_n):
    df_less_n['TEXT']=df_less_n['TEXT'].fillna(' ')
    df_less_n['TEXT']=df_less_n['TEXT'].str.replace('\n',' ', regex=False)
    df_less_n['TEXT']=df_less_n['TEXT'].str.replace('\r',' ', regex=False)
    df_less_n['TEXT']=df_less_n['TEXT'].apply(str.strip)
    df_less_n['TEXT']=df_less_n['TEXT'].str.lower()

    df_less_n['TEXT']=df_less_n['TEXT'].apply(lambda x: preprocess1(x))

    #to get 318 words chunks for readmission tasks
    df_len = len(df_less_n)
    new_rows = []

    for i in tqdm(range(df_len)):
        x = df_less_n.TEXT.iloc[i].split()
        n = int(len(x)/318)
        label = df_less_n.OUTPUT_LABEL.iloc[i]
        hadm_id = df_less_n.HADM_ID.iloc[i]

        for j in range(n):
            new_rows.append({
                'TEXT': ' '.join(x[j*318:(j+1)*318]),
                'Label': label,
                'ID': hadm_id
            })
        if len(x) % 318 > 10:
            new_rows.append({
                'TEXT': ' '.join(x[-(len(x)%318):]),
                'Label': label,
                'ID': hadm_id
            })

    return pd.DataFrame(new_rows)

df_discharge = preprocessing(df_adm_disch)



KeyboardInterrupt: 

In [24]:
import re
import pandas as pd
from tqdm import tqdm

def preprocess1(x):
    y=re.sub(r'\[(.*?)\]','',x) #remove de-identified brackets
    y=re.sub(r'[0-9]+\.','',y) #remove 1.2. since the segmenter segments based on this
    y=re.sub(r'dr\.','doctor',y)
    y=re.sub(r'm\.d\.','md',y)
    y=re.sub('admission date:','',y)
    y=re.sub('discharge date:','',y)
    y=re.sub('--|__|==','',y)
    return y

print("Starting text preprocessing...")

df_adm_disch['TEXT']=df_adm_disch['TEXT'].fillna(' ')
df_adm_disch['TEXT']=df_adm_disch['TEXT'].str.replace('\n',' ', regex=False)
df_adm_disch['TEXT']=df_adm_disch['TEXT'].str.replace('\r',' ', regex=False)
df_adm_disch['TEXT']=df_adm_disch['TEXT'].apply(str.strip)
df_adm_disch['TEXT']=df_adm_disch['TEXT'].str.lower()

# Apply regex preprocessing with progress bar
tqdm.pandas(desc="Applying Regex Preprocessing")
df_adm_disch['TEXT']=df_adm_disch['TEXT'].progress_apply(lambda x: preprocess1(x))

print("Preprocessing complete! You can now run your chunking and saving cell.")

Starting text preprocessing...


Applying Regex Preprocessing: 100%|██████████| 323603/323603 [02:08<00:00, 2510.76it/s]


Preprocessing complete! You can now run your chunking and saving cell.


In [25]:
import math
import os
from tqdm import tqdm

# Calculate rows and unique subjects with empty/null text
empty_mask = df_adm_disch['TEXT'].isnull() | (df_adm_disch['TEXT'].astype(str).str.strip() == '')
num_empty_rows = empty_mask.sum()
num_empty_subjects = df_adm_disch[empty_mask]['SUBJECT_ID'].nunique()

print(f'Number of rows with empty text: {num_empty_rows}')
print(f'Number of unique subject IDs with empty text: {num_empty_subjects}\n')
df_adm_disch.dropna(subset=['TEXT'], inplace=True)

# Create the output directory if it doesn't exist
output_dir = '/content/drive/MyDrive/NarrativeGuard/mimic-discharge-summaries/discharge_split'

chunk_size = 1000
total_rows = len(df_adm_disch)
num_chunks = math.ceil(total_rows / chunk_size)

print(f'Total rows in df_adm_disch: {total_rows}')
print(f'Splitting into {num_chunks} files with {chunk_size} rows per file.')

for i in tqdm(range(num_chunks), desc='Saving chunks'):
    start_row = i * chunk_size
    end_row = min((i + 1) * chunk_size, total_rows)
    chunk_df = df_adm_disch.iloc[start_row:end_row]

    # Generate filename based on row range (1-indexed for clarity)
    filename = f'adm_disch_comb_{start_row + 1}_{end_row}k.csv'
    output_path = os.path.join(output_dir, filename)

    chunk_df.to_csv(output_path, index=False)
    # print(f'Saved {output_path}') # Optional: comment out to reduce output noise

print('\nAll chunks saved successfully!')

Number of rows with empty text: 0
Number of unique subject IDs with empty text: 0

Total rows in df_adm_disch: 323603
Splitting into 324 files with 1000 rows per file.


Saving chunks: 100%|██████████| 324/324 [02:27<00:00,  2.19it/s]


All chunks saved successfully!
